 # <center> **Анализ и прогнозирование временных рядов болезни Альцгеймера** </center>

### **Часть 1**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly


from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error


plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

In [ ]:
def load_data_from_file(file_path='Data.xlsx'):
    
    df = pd.read_excel(file_path)
    
    
    df = df.rename(columns={
        'Количество случаев заболеваемости': 'Количество_случаев',
        'Количество смертей': 'Количество_смертей'
    })
    
    
    df['Распространённость'] = df['Приток'] - df['Отток']
    

    df = df.sort_values(['Страна', 'Год'])
    
    return df


df_raw = load_data_from_file('Data.xlsx')


print(f"Форма данных: {df_raw.shape[0]} строк, {df_raw.shape[1]} колонок")
print(f"Период: {df_raw['Год'].min()} - {df_raw['Год'].max()}")
print(f"Количество стран: {df_raw['Страна'].nunique()}")
print(f"\nСтраны в данных:")
for country in df_raw['Страна'].unique():
    print(f"  - {country}")
print("\nПервые 5 строк:")
print(df_raw.head())

In [ ]:
def prepare_prophet_data(df, country):
    
    country_df = df[df['Страна'] == country].copy()
    prophet_df = pd.DataFrame({
        'ds': pd.to_datetime(country_df['Год'], format='%Y'),
        'y': country_df['Распространённость']
    })
  
    prophet_df = prophet_df.set_index('ds').asfreq('YS').reset_index()
   
    if prophet_df['y'].isnull().any():
        prophet_df['y'] = prophet_df['y'].interpolate(method='linear')
    return prophet_df


example_country = df_raw['Страна'].iloc[0]
prophet_data = prepare_prophet_data(df_raw, example_country)

print(f"Пример для страны: {example_country}")
print(prophet_data.head(10))


fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(prophet_data['ds'], prophet_data['y'], 'b-', linewidth=2)
ax.set_title(f'Временной ряд распространённости болезни Альцгеймера в стране {example_country}', fontsize=14)
ax.set_xlabel('Год')
ax.set_ylabel('Количество человек')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("="*50)
print("ПРОВЕРКА ДАННЫХ")
print("="*50)

italy_data = df_raw[df_raw['Страна'] == 'Италия'].copy()
print(f"\nИталия: {len(italy_data)} лет данных")
print(f"Диапазон: {italy_data['Год'].min()} - {italy_data['Год'].max()}")
print(f"Распространённость в 1990: {italy_data[italy_data['Год']==1990]['Распространённость'].values[0]:,.0f}")
print(f"Распространённость в 2021: {italy_data[italy_data['Год']==2021]['Распространённость'].values[0]:,.0f}")
print(f"Рост: {italy_data[italy_data['Год']==2021]['Распространённость'].values[0] / italy_data[italy_data['Год']==1990]['Распространённость'].values[0]:.1f}x")

In [ ]:
countries = sorted(df_raw['Страна'].unique().tolist())
print("Список стран для анализа:")
print("="*40)
for i, country in enumerate(countries, 1):
    data_years = len(df_raw[df_raw['Страна'] == country])
    print(f"{i:2}. {country:<15} ({data_years} лет данных)")

print(f"\nВсего стран: {len(countries)}")

In [ ]:
def train_basic_prophet(df, country):
    
    prophet_df = prepare_prophet_data(df, country)
    
    
    model = Prophet(
        growth='linear',
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.95
    )
    
    
    model.fit(prophet_df)
    
    
    future = model.make_future_dataframe(periods=10, freq='YS')
    forecast = model.predict(future)
    
    return model, forecast, prophet_df


model_de, forecast_de, history_de = train_basic_prophet(df_raw, 'Германия')

print(" Модель Prophet обучена для Германии")
print(f"Период прогноза: {forecast_de['ds'].min().year} - {forecast_de['ds'].max().year}")
forecast_de[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10)

In [ ]:
def plot_basic_forecast(model, forecast, country):
    
    fig1 = model.plot(forecast)
    plt.title(f'Прогноз распространённости болезни Альцгеймера в {country}', fontsize=14)
    plt.xlabel('Год')
    plt.ylabel('Количество человек')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    fig2 = model.plot_components(forecast)
    plt.suptitle(f'Компоненты модели Prophet для {country}', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_basic_forecast(model_de, forecast_de, 'Германия')

In [ ]:
countries = df_raw['Страна'].unique().tolist()
results_semester1 = []

for country in countries:
    print(f"\nОбработка: {country}")
    
    try:
        prophet_df = prepare_prophet_data(df_raw, country)
        
        model = Prophet(yearly_seasonality=True, interval_width=0.95)
        model.fit(prophet_df)
        
        future = model.make_future_dataframe(periods=10, freq='YS')
        forecast = model.predict(future)
        
        
        forecast_2031 = forecast[forecast['ds'].dt.year == 2031]['yhat'].values[0]
        
      
        from sklearn.metrics import mean_absolute_error, mean_squared_error
        y_true = prophet_df['y'].values
        y_pred_train = forecast[forecast['ds'].isin(prophet_df['ds'])]['yhat'].values
        
        mae = mean_absolute_error(y_true, y_pred_train)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred_train))
        mape = np.mean(np.abs((y_true - y_pred_train) / (y_true + 1e-8))) * 100
        
        results_semester1.append({
            'Страна': country,
            'MAE': round(mae, 2),
            'RMSE': round(rmse, 2),
            'MAPE': round(mape, 2),
            'Прогноз_2031': round(forecast_2031, 0)
        })
        
        print(f"  ✓ MAPE = {mape:.2f}%")
        
    except Exception as e:
        print(f"  ✗ Ошибка: {e}")
        results_semester1.append({
            'Страна': country,
            'MAE': None,
            'RMSE': None,
            'MAPE': None,
            'Прогноз_2031': None
        })


results_df_s1 = pd.DataFrame(results_semester1)
print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ ПЕРВОГО СЕМЕСТРА")
print("="*60)
results_df_s1

In [ ]:
results = pd.DataFrame({
    'Страна': ['Австрия', 'Бельгия', 'Германия', 'Греция', 'Италия', 
               'Нидерланды', 'Португалия', 'Сингапур', 'Финляндия', 
               'Швейцария', 'Швеция'],
    'MAPE': [0.17, 0.03, 0.07, 0.03, 0.76, 0.11, 0.08, 1.23, 0.09, 0.00, 0.11],
    'Прогноз_2031': [21665, 936199, 9061035, 979991, 5159734, 
                     1218580, 889571, 146868, 403157, 11655, 942862]
})

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax1 = axes[0]
colors = ['green' if x < 0.2 else 'orange' if x < 0.8 else 'red' for x in results['MAPE']]
bars = ax1.barh(results['Страна'], results['MAPE'], color=colors)
ax1.set_xlabel('MAPE (%)', fontsize=12)
ax1.set_title('Точность модели по странам (чем меньше, тем лучше)', fontsize=14)
ax1.axvline(x=0.2, color='green', linestyle='--', alpha=0.5, label='Отлично (<0.2%)')
ax1.axvline(x=0.8, color='orange', linestyle='--', alpha=0.5, label='Требует внимания (>0.8%)')
ax1.legend()


for bar, val in zip(bars, results['MAPE']):
    ax1.text(val + 0.02, bar.get_y() + bar.get_height()/2, f'{val}%', va='center')

ax2 = axes[1]
results_sorted = results.sort_values('Прогноз_2031')
bars2 = ax2.barh(results_sorted['Страна'], results_sorted['Прогноз_2031'], color='steelblue')
ax2.set_xlabel('Прогноз на 2031 (человек)', fontsize=12)
ax2.set_title('Прогнозируемое число больных на 2031 год', fontsize=14)
ax2.set_xscale('log')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
results_df_s1.to_csv('results_semester1.csv', index=False)



fig, ax = plt.subplots(figsize=(12, 6))
countries_plot = results_df_s1['Страна'].tolist()
mape_values = results_df_s1['MAPE'].tolist()

bars = ax.barh(countries_plot, mape_values, color='steelblue')
ax.set_xlabel('MAPE (%)', fontsize=12)
ax.set_title('Точность моделей Prophet (первый семестр)', fontsize=14)
ax.grid(True, alpha=0.3, axis='x')

for bar, val in zip(bars, mape_values):
    if val:
        ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center')

plt.tight_layout()
plt.show()

In [ ]:
countries = sorted(df_raw['Страна'].unique().tolist())
print(f"Стран для анализа: {len(countries)}")
print(f"Список: {countries}")

import os
os.makedirs('forecast_plots', exist_ok=True)

metrics_dict = {}

for i, country in enumerate(countries, 1):
    print(f"\n{i}/{len(countries)}: {country}")
    
    try:
        prophet_df = prepare_prophet_data(df_raw, country)
        tune = country in ['Италия', 'Сингапур']
        
        if tune:
            model = Prophet(changepoint_prior_scale=1.0, yearly_seasonality=True)
        else:
            model = Prophet(yearly_seasonality=True)
        
        model.fit(prophet_df)
        future = model.make_future_dataframe(periods=10, freq='YS')
        forecast = model.predict(future)
        
        from sklearn.metrics import mean_absolute_error, mean_squared_error
        y_true = prophet_df['y'].values
        y_pred = forecast[forecast['ds'].isin(prophet_df['ds'])]['yhat'].values
        
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
        
        forecast_2031 = forecast[forecast['ds'].dt.year == 2031]['yhat'].values[0] if len(forecast[forecast['ds'].dt.year == 2031]) > 0 else None
        
        metrics_dict[country] = {
            'MAE': mae,
            'RMSE': rmse,
            'MAPE': mape,
            'Прогноз_2031': forecast_2031
        }
        
        
        fig, ax = plt.subplots(figsize=(14, 7))
        
    
        ax.plot(prophet_df['ds'], prophet_df['y'], 'b-', linewidth=2.5, label='Исторические данные (1990-2021)')
        
        
        forecast_future = forecast[forecast['ds'] > prophet_df['ds'].max()]
        ax.plot(forecast_future['ds'], forecast_future['yhat'], 'r--', linewidth=2, label='Прогноз (2022-2031)')
        
    
        ax.fill_between(forecast_future['ds'], 
                         forecast_future['yhat_lower'], 
                         forecast_future['yhat_upper'], 
                         color='red', alpha=0.2, label='95% доверительный интервал')
        
        
        ax.set_title(f'{country} | MAPE = {mape:.2f}% | Прогноз на 2031: {forecast_2031:,.0f}', 
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Год', fontsize=12)
        ax.set_ylabel('Количество человек', fontsize=12)
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
        
        
        from matplotlib.ticker import FuncFormatter
        def format_y(x, p):
            if x >= 1e6:
                return f'{x/1e6:.1f}M'
            elif x >= 1e3:
                return f'{x/1e3:.0f}K'
            return f'{x:.0f}'
        ax.yaxis.set_major_formatter(FuncFormatter(format_y))
        
        plt.tight_layout()
        
        
        safe_name = country.replace(' ', '_').replace('/', '_')
        plt.savefig(f'forecast_plots/{safe_name}.png', dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"  MAPE = {mape:.2f}%, Прогноз 2031 = {forecast_2031:,.0f}")
        
    except Exception as e:
        print(f"  Ошибка: {e}")


### **Часть 2**

In [ ]:
def train_test_split_time(df, train_end_year=2015):
    """Корректное разбиение временного ряда"""
    train = df[df['ds'].dt.year <= train_end_year].copy()
    test = df[df['ds'].dt.year > train_end_year].copy()
    return train, test


prophet_df = prepare_prophet_data(df_raw, 'Италия')
train, test = train_test_split_time(prophet_df)

print(f"Обучающий период: {train['ds'].min().year} - {train['ds'].max().year} ({len(train)} лет)")
print(f"Тестовый период: {test['ds'].min().year} - {test['ds'].max().year} ({len(test)} лет)")


fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train['ds'], train['y'], 'b-', linewidth=2, label='Train (1990-2015)')
ax.plot(test['ds'], test['y'], 'g-', linewidth=2, label='Test (2016-2021)')
ax.axvline(x=pd.Timestamp('2015-01-01'), color='red', linestyle='--', alpha=0.7, label='Граница разделения')
ax.set_title('Разбиение данных на обучающую и тестовую выборки', fontsize=14)
ax.set_xlabel('Год')
ax.set_ylabel('Количество человек')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def train_tuned_prophet(train_df, test_df, country):
    """Prophet с настройкой для проблемных стран"""
    
    if country in ['Италия', 'Сингапур']:
        model = Prophet(
            changepoint_prior_scale=1.0,      
            seasonality_prior_scale=10.0,
            yearly_seasonality=True,
            interval_width=0.95
        )
    else:
        model = Prophet(
            changepoint_prior_scale=0.5,
            yearly_seasonality=True,
            interval_width=0.95
        )
    
    model.fit(train_df)
    
    
    forecast_test = model.predict(test_df[['ds']])
    
    
    future = model.make_future_dataframe(periods=10, freq='YS')
    forecast_future = model.predict(future)
    
    return model, forecast_test, forecast_future

In [ ]:
def find_best_arima_order(series, max_p=3, max_q=3):
    """Автоматический подбор параметров ARIMA"""
    result = adfuller(series.dropna())
    d = 0 if result[1] < 0.05 else 1
    
    best_aic = np.inf
    best_order = None
    
    for p in range(max_p + 1):
        for q in range(max_q + 1):
            try:
                model = ARIMA(series, order=(p, d, q))
                fitted = model.fit()
                if fitted.aic < best_aic:
                    best_aic = fitted.aic
                    best_order = (p, d, q)
            except:
                continue
    return best_order

def calculate_metrics(actual, predicted):
    """Расчёт всех метрик качества"""
    actual = np.array(actual)
    predicted = np.array(predicted)
    
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / (actual + 1e-8))) * 100
    
    return {
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'MAPE': round(mape, 2)
    }

In [ ]:
comparison_results = []

for country in countries:
    print(f"\nАнализ: {country}")
    
    try:
        prophet_df = prepare_prophet_data(df_raw, country)
        train, test = train_test_split_time(prophet_df)
        
        if len(test) < 2:
            print(f"  Недостаточно тестовых данных")
            continue
        
        
        tune = country in ['Италия', 'Сингапур']
        if tune:
            model = Prophet(changepoint_prior_scale=1.0, yearly_seasonality=True)
        else:
            model = Prophet(yearly_seasonality=True)
        
        model.fit(train)
        forecast_test = model.predict(test[['ds']])
        
     
        train_series = train.set_index('ds')['y']
        test_series = test.set_index('ds')['y']
        
        best_order = find_best_arima_order(train_series)
        if best_order:
            arima_model = ARIMA(train_series, order=best_order)
            fitted_arima = arima_model.fit()
            arima_forecast = fitted_arima.forecast(steps=len(test_series))
        else:
            arima_forecast = [train_series.iloc[-1]] * len(test_series)
        
        
        actual = test['y'].values
        prophet_metrics = calculate_metrics(actual, forecast_test['yhat'].values[:len(actual)])
        arima_metrics = calculate_metrics(actual, arima_forecast)
        
        comparison_results.append({
            'Страна': country,
            'Prophet_MAPE': prophet_metrics['MAPE'],
            'ARIMA_MAPE': arima_metrics['MAPE'],
            'Лучшая': 'Prophet' if prophet_metrics['MAPE'] < arima_metrics['MAPE'] else 'ARIMA'
        })
        
        print(f"  Prophet MAPE: {prophet_metrics['MAPE']:.2f}%")
        print(f"  ARIMA MAPE: {arima_metrics['MAPE']:.2f}%")
        print(f"  Лучшая: {comparison_results[-1]['Лучшая']}")
        
    except Exception as e:
        print(f"  Ошибка: {e}")

comparison_df = pd.DataFrame(comparison_results)
print("\n" + "="*60)
print("СРАВНЕНИЕ PROPHET vs ARIMA")
print("="*60)
print(comparison_df.to_string(index=False))


comparison_df.to_csv('comparison_prophet_arima.csv', index=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
x = np.arange(len(comparison_df))
width = 0.35

bars1 = ax1.bar(x - width/2, comparison_df['Prophet_MAPE'], width, label='Prophet', color='steelblue')
bars2 = ax1.bar(x + width/2, comparison_df['ARIMA_MAPE'], width, label='ARIMA', color='coral')

ax1.set_xlabel('Страна', fontsize=12)
ax1.set_ylabel('MAPE (%)', fontsize=12)
ax1.set_title('Сравнение точности прогнозов (тест 2016-2021)', fontsize=14)
ax1.set_xticks(x)
ax1.set_xticklabels(comparison_df['Страна'], rotation=45, ha='right')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')


ax2 = axes[1]
better_counts = comparison_df['Лучшая'].value_counts()
colors2 = ['green' if i == 'Prophet' else 'red' for i in better_counts.index]
bars3 = ax2.bar(better_counts.index, better_counts.values, color=colors2)
ax2.set_ylabel('Количество стран', fontsize=12)
ax2.set_title('Prophet vs ARIMA: кто точнее?', fontsize=14)
for bar, val in zip(bars3, better_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, str(val), ha='center', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('prophet_vs_arima.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:

import os
os.makedirs('forecast_plots', exist_ok=True)

for country in countries:
    print(f"График для: {country}")
    
    try:
        prophet_df = prepare_prophet_data(df_raw, country)
        tune = country in ['Италия', 'Сингапур']
        
        if tune:
            model = Prophet(changepoint_prior_scale=1.0, yearly_seasonality=True)
        else:
            model = Prophet(yearly_seasonality=True)
        
        model.fit(prophet_df)
        future = model.make_future_dataframe(periods=10, freq='YS')
        forecast = model.predict(future)
        
       
        y_true = prophet_df['y'].values
        y_pred = forecast[forecast['ds'].isin(prophet_df['ds'])]['yhat'].values
        mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
        
        
        fig, ax = plt.subplots(figsize=(14, 7))
        
        ax.plot(prophet_df['ds'], prophet_df['y'], 'b-', linewidth=2.5, label='Исторические данные (1990-2021)')
        
        forecast_future = forecast[forecast['ds'] > prophet_df['ds'].max()]
        ax.plot(forecast_future['ds'], forecast_future['yhat'], 'r--', linewidth=2, label='Прогноз (2022-2031)')
        ax.fill_between(forecast_future['ds'], forecast_future['yhat_lower'], forecast_future['yhat_upper'], 
                         color='red', alpha=0.2, label='95% доверительный интервал')
        
        forecast_2031 = forecast[forecast['ds'].dt.year == 2031]['yhat'].values[0]
        ax.set_title(f'{country} | MAPE = {mape:.2f}% | Прогноз на 2031: {forecast_2031:,.0f}', 
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Год')
        ax.set_ylabel('Количество человек')
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
        
       
        from matplotlib.ticker import FuncFormatter
        def format_y(x, p):
            if x >= 1e6:
                return f'{x/1e6:.1f}M'
            elif x >= 1e3:
                return f'{x/1e3:.0f}K'
            return f'{x:.0f}'
        ax.yaxis.set_major_formatter(FuncFormatter(format_y))
        
        plt.tight_layout()
        safe_name = country.replace(' ', '_')
        plt.savefig(f'forecast_plots/{safe_name}.png', dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"   MAPE = {mape:.2f}%")
        
    except Exception as e:
        print(f"   Ошибка: {e}")


In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))

colors = plt.cm.tab20(np.linspace(0, 1, len(countries)))

for i, country in enumerate(countries):
    prophet_df = prepare_prophet_data(df_raw, country)
    ax.plot(prophet_df['ds'], prophet_df['y'], linewidth=2, label=country, color=colors[i])

ax.set_title('Сравнение временных рядов по странам', fontsize=14, fontweight='bold')
ax.set_xlabel('Год', fontsize=12)
ax.set_ylabel('Количество человек', fontsize=12)
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)

from matplotlib.ticker import FuncFormatter
def format_y(x, p):
    if x >= 1e6:
        return f'{x/1e6:.0f}M'
    elif x >= 1e3:
        return f'{x/1e3:.0f}K'
    return f'{x:.0f}'
ax.yaxis.set_major_formatter(FuncFormatter(format_y))

plt.tight_layout()
plt.savefig('forecast_plots/all_countries_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
final_results = []

for country in countries:
    prophet_df = prepare_prophet_data(df_raw, country)
    train, test = train_test_split_time(prophet_df)
    
    # Prophet
    tune = country in ['Италия', 'Сингапур']
    model = Prophet(changepoint_prior_scale=1.0 if tune else 0.5, yearly_seasonality=True)
    model.fit(train)
    forecast_test = model.predict(test[['ds']])
    
    prophet_mape = calculate_metrics(test['y'].values, forecast_test['yhat'].values[:len(test)])['MAPE']
    
    # ARIMA
    train_series = train.set_index('ds')['y']
    test_series = test.set_index('ds')['y']
    best_order = find_best_arima_order(train_series)
    if best_order:
        arima_model = ARIMA(train_series, order=best_order)
        arima_forecast = arima_model.fit().forecast(steps=len(test_series))
        arima_mape = calculate_metrics(test_series.values, arima_forecast)['MAPE']
    else:
        arima_mape = None
    
    final_results.append({
        'Страна': country,
        'Prophet MAPE (%)': prophet_mape,
        'ARIMA MAPE (%)': arima_mape if arima_mape else 'N/A',
        'Лучшая модель': 'Prophet' if arima_mape and prophet_mape < arima_mape else ('ARIMA' if arima_mape else 'Prophet')
    })

final_df = pd.DataFrame(final_results)
print("="*70)
print("ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*70)
print(final_df.to_string(index=False))

final_df.to_csv('final_results.csv', index=False)
